In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
df = pd.read_parquet("../../data/processed/site_features/property_site_features_v1_sample_scored.parquet")
print(len(df))
df.head()

49950


,RID,gurasid,propertytype,valnetpropertystatus,valnetpropertytype,dissolveparcelcount,valnetlotcount,propid,superlot,address,...,log_distance_to_station_m,station_access_score,station_distance_bucket,single_dwelling_rebuild_score,assembly_opportunity_score,granny_flat_score,land_bank_hold_score,townhouse_multi_dwelling_score,low_rise_apartment_score,dual_occupancy_score
0,260,49494103,1,2.0,2.0,1,1,1341,N,44 NORTHCOTE STREET ABERDARE,...,9.806382,0.000055,>10km,70.0,12.0,70.0,45.0,22.0,1.0,70.0
1,319,49494159,1,2.0,2.0,1,1,3169,N,10 MEREWETHER CLOSE NORTH ROTHBURY,...,8.129650,0.000295,2km-5km,45.2,19.6,53.5,29.6,19.0,13.5,38.8
2,327,49494166,1,2.0,2.0,1,1,3682,N,112 MATHIESON STREET BELLBIRD HEIGHTS,...,9.958687,0.000047,>10km,70.0,12.0,70.0,45.0,22.0,1.0,60.0
3,338,49494176,1,2.0,2.0,1,1,3958,N,584 WOLLOMBI ROAD BELLBIRD,...,10.104941,0.000041,>10km,32.0,6.0,11.5,17.0,1.0,0.5,11.5
4,359,49494197,1,2.0,2.0,1,1,4739,N,41 RUSSELL STREET BRANXTON,...,5.679883,0.003414,<=800m,61.0,43.0,62.5,78.0,65.0,47.5,64.0


In [3]:
def lot_size_band(area):
    if pd.isna(area):
        return "unknown"
    if area < 500:
        return "xs"
    elif area < 1000:
        return "s"
    elif area < 2000:
        return "m"
    elif area < 5000:
        return "l"
    else:
        return "xl"

df["lot_size_band"] = df["lot_size_proxy_sqm"].apply(lot_size_band)
df["lot_size_band"].value_counts(dropna=False)

lot_size_band
s     14892
m     13684
xl    10995
l      6816
xs     3563
Name: count, dtype: int64

In [4]:
def station_distance_band(dist):
    if pd.isna(dist):
        return "unknown"
    if dist <= 800:
        return "within_800m"
    elif dist <= 2000:
        return "800m_2km"
    elif dist <= 5000:
        return "2km_5km"
    elif dist <= 10000:
        return "5km_10km"
    else:
        return "over_10km"

df["station_distance_band"] = df["distance_to_station_m"].apply(station_distance_band)
df["station_distance_band"].value_counts(dropna=False)

station_distance_band
over_10km      14495
800m_2km       10978
2km_5km        10817
within_800m    10242
5km_10km        3418
Name: count, dtype: int64

In [5]:
def constraint_severity(row):
    score = 0

    if int(row.get("heritage_flag", 0) or 0) == 1:
        score += 2
        sig = row.get("heritage_max_significance")
        if pd.notna(sig) and str(sig) in {"State", "National"}:
            score += 1

    if int(row.get("flood_flag", 0) or 0) == 1:
        score += 2

    bushfire = int(row.get("bushfire_risk_level", 0) or 0)
    score += bushfire

    if score == 0:
        return "low"
    elif score <= 2:
        return "moderate"
    elif score <= 4:
        return "high"
    return "severe"

df["constraint_severity_band"] = df.apply(constraint_severity, axis=1)
df["constraint_severity_band"].value_counts(dropna=False)

constraint_severity_band
low         33465
moderate    11830
high         4277
severe        378
Name: count, dtype: int64

In [6]:
def zoning_band(z):
    if pd.isna(z):
        return "unknown"

    z = str(z)

    if z in {"MU1", "R4", "SP5", "E1", "E2", "E3"}:
        return "high_dev"
    elif z in {"R3", "R1"}:
        return "mid_dev"
    elif z in {"R2", "R5", "RU5"}:
        return "low_dev"
    elif z in {"C2", "C3", "C4", "RE1", "RU1", "RU4", "SP2", "W1", "W2", "W3", "W4"}:
        return "restricted"
    else:
        return "other"

df["zoning_band"] = df["primary_zoning_code"].apply(zoning_band)
df["zoning_band"].value_counts(dropna=False)

zoning_band
low_dev       20167
mid_dev       11214
restricted     8321
high_dev       7362
other          2753
unknown         133
Name: count, dtype: int64

In [7]:
strategy_score_cols = [
    "single_dwelling_rebuild_score",
    "assembly_opportunity_score",
    "granny_flat_score",
    "land_bank_hold_score",
    "townhouse_multi_dwelling_score",
    "low_rise_apartment_score",
    "dual_occupancy_score",
]

df["top_strategy"] = df[strategy_score_cols].idxmax(axis=1).str.replace("_score", "", regex=False)
df["top_strategy_score"] = df[strategy_score_cols].max(axis=1)

df[["top_strategy", "top_strategy_score"]].head()

,top_strategy,top_strategy_score
0,single_dwelling_rebuild,70.0
1,granny_flat,53.5
2,single_dwelling_rebuild,70.0
3,single_dwelling_rebuild,32.0
4,land_bank_hold,78.0


In [8]:
def build_site_summary_text(row):
    parts = []

    z = row.get("primary_zoning_code")
    if pd.notna(z):
        parts.append(f"zoning {z}")

    lot_band = row.get("lot_size_band")
    if pd.notna(lot_band):
        parts.append(f"lot size band {lot_band}")

    if int(row.get("mixed_zoning_flag", 0) or 0) == 1:
        parts.append("mixed zoning context")

    if int(row.get("heritage_flag", 0) or 0) == 1:
        sig = row.get("heritage_max_significance")
        if pd.notna(sig):
            parts.append(f"heritage constraint {str(sig).lower()} significance")
        else:
            parts.append("heritage constraint")

    if int(row.get("flood_flag", 0) or 0) == 1:
        flood_class = row.get("primary_flood_class")
        if pd.notna(flood_class):
            parts.append(f"flood constraint {str(flood_class).lower()}")
        else:
            parts.append("flood constraint")
    else:
        parts.append("no flood constraint")

    bushfire_level = int(row.get("bushfire_risk_level", 0) or 0)
    if bushfire_level == 0:
        parts.append("no bushfire constraint")
    else:
        parts.append(f"bushfire risk level {bushfire_level}")

    if int(row.get("within_800m_catchment", 0) or 0) == 1:
        parts.append("within 800m of rail or metro")
    else:
        dist_band = row.get("station_distance_band")
        if pd.notna(dist_band):
            parts.append(f"station distance band {dist_band}")

    top_strategy = row.get("top_strategy")
    if pd.notna(top_strategy):
        parts.append(f"top strategy signal {top_strategy}")

    return " | ".join(parts)

df["site_summary_text"] = df.apply(build_site_summary_text, axis=1)
df[["RID", "address", "site_summary_text"]].head(10)

,RID,address,site_summary_text
0,260,44 NORTHCOTE STREET ABERDARE,zoning R2 | lot size band m | no flood constra...
1,319,10 MEREWETHER CLOSE NORTH ROTHBURY,zoning R5 | lot size band xl | no flood constr...
2,327,112 MATHIESON STREET BELLBIRD HEIGHTS,zoning R2 | lot size band m | no flood constra...
3,338,584 WOLLOMBI ROAD BELLBIRD,zoning SP2 | lot size band s | mixed zoning co...
4,359,41 RUSSELL STREET BRANXTON,zoning R3 | lot size band m | no flood constra...
5,400,14 FOSTER STREET CESSNOCK,zoning R3 | lot size band s | no flood constra...
6,456,13 WILLIAM STREET CESSNOCK,zoning R3 | lot size band s | no flood constra...
7,606,761 WATAGAN CREEK ROAD WATAGAN,zoning C2 | lot size band xl | mixed zoning co...
8,913,4 CORAL CRESCENT PEARL BEACH,zoning R2 | lot size band m | mixed zoning con...
9,951,65 DONALD AVENUE UMINA BEACH,zoning R2 | lot size band s | no flood constra...


In [9]:
def build_candidate_text(row):
    parts = []

    z = row.get("primary_zoning_code")
    if pd.notna(z):
        parts.append(f"The site has {z} zoning.")

    lot_band = row.get("lot_size_band")
    if pd.notna(lot_band):
        parts.append(f"The lot size band is {lot_band}.")

    if int(row.get("within_800m_catchment", 0) or 0) == 1:
        parts.append("It is within 800m of rail or metro access.")
    else:
        dist = row.get("distance_to_station_m")
        if pd.notna(dist):
            parts.append(f"The nearest rail or metro access is about {int(dist)} metres away.")

    if int(row.get("heritage_flag", 0) or 0) == 1:
        parts.append("Heritage constraints are present.")

    if int(row.get("flood_flag", 0) or 0) == 1:
        parts.append("Flood-related planning constraints are present.")

    bushfire = int(row.get("bushfire_risk_level", 0) or 0)
    if bushfire > 0:
        parts.append(f"Bushfire risk level is {bushfire}.")

    top_strategy = row.get("top_strategy")
    if pd.notna(top_strategy):
        parts.append(f"The strongest current strategy signal is {top_strategy.replace('_', ' ')}.")

    return " ".join(parts)

df["candidate_text"] = df.apply(build_candidate_text, axis=1)
df[["RID", "candidate_text"]].head(10)

,RID,candidate_text
0,260,The site has R2 zoning. The lot size band is m...
1,319,The site has R5 zoning. The lot size band is x...
2,327,The site has R2 zoning. The lot size band is m...
3,338,The site has SP2 zoning. The lot size band is ...
4,359,The site has R3 zoning. The lot size band is m...
5,400,The site has R3 zoning. The lot size band is s...
6,456,The site has R3 zoning. The lot size band is s...
7,606,The site has C2 zoning. The lot size band is x...
8,913,The site has R2 zoning. The lot size band is m...
9,951,The site has R2 zoning. The lot size band is s...


In [10]:
candidate_cols = [
    "RID",
    "address",
    "primary_zoning_code",
    "primary_zoning_class",
    "zoning_band",
    "mixed_zoning_flag",
    "lot_size_proxy_sqm",
    "lot_size_band",
    "heritage_flag",
    "heritage_max_significance",
    "bushfire_flag",
    "bushfire_risk_level",
    "flood_flag",
    "primary_flood_class",
    "distance_to_station_m",
    "within_800m_catchment",
    "station_distance_band",
    "station_access_score",
    "constraint_severity_band",
    "top_strategy",
    "top_strategy_score",
    "site_summary_text",
    "candidate_text",
]
candidate_sites = df[candidate_cols].copy()
candidate_sites.head()

,RID,address,primary_zoning_code,primary_zoning_class,zoning_band,mixed_zoning_flag,lot_size_proxy_sqm,lot_size_band,heritage_flag,heritage_max_significance,...,primary_flood_class,distance_to_station_m,within_800m_catchment,station_distance_band,station_access_score,constraint_severity_band,top_strategy,top_strategy_score,site_summary_text,candidate_text
0,260,44 NORTHCOTE STREET ABERDARE,R2,Low Density Residential,low_dev,0,1439.819123,m,0,None,...,None,18148.199692,0,over_10km,0.000055,low,single_dwelling_rebuild,70.0,zoning R2 | lot size band m | no flood constra...,The site has R2 zoning. The lot size band is m...
1,319,10 MEREWETHER CLOSE NORTH ROTHBURY,R5,Large Lot Residential,low_dev,0,25509.139491,xl,0,None,...,None,3392.611518,0,2km_5km,0.000295,high,granny_flat,53.5,zoning R5 | lot size band xl | no flood constr...,The site has R5 zoning. The lot size band is x...
2,327,112 MATHIESON STREET BELLBIRD HEIGHTS,R2,Low Density Residential,low_dev,0,1129.358164,m,0,None,...,None,21134.025534,0,over_10km,0.000047,low,single_dwelling_rebuild,70.0,zoning R2 | lot size band m | no flood constra...,The site has R2 zoning. The lot size band is m...
3,338,584 WOLLOMBI ROAD BELLBIRD,SP2,Infrastructure,restricted,1,987.808843,s,0,None,...,None,24462.595670,0,over_10km,0.000041,moderate,single_dwelling_rebuild,32.0,zoning SP2 | lot size band s | mixed zoning co...,The site has SP2 zoning. The lot size band is ...
4,359,41 RUSSELL STREET BRANXTON,R3,Medium Density Residential,mid_dev,0,1347.510500,m,0,None,...,None,291.915278,1,within_800m,0.003414,low,land_bank_hold,78.0,zoning R3 | lot size band m | no flood constra...,The site has R3 zoning. The lot size band is m...


In [11]:
Path("../../data/processed/retrieval").mkdir(parents=True, exist_ok=True)

candidate_sites.to_parquet(
    "../../data/processed/retrieval/candidate_sites.parquet",
    index=False
)

In [12]:
candidate_sites.sample(20, random_state=42)

,RID,address,primary_zoning_code,primary_zoning_class,zoning_band,mixed_zoning_flag,lot_size_proxy_sqm,lot_size_band,heritage_flag,heritage_max_significance,...,primary_flood_class,distance_to_station_m,within_800m_catchment,station_distance_band,station_access_score,constraint_severity_band,top_strategy,top_strategy_score,site_summary_text,candidate_text
8194,893252,225 WATKINS ROAD WANGI WANGI,R2,Low Density Residential,low_dev,1,1.215176e+03,m,0,None,...,None,9317.476976,0,5km_10km,0.000107,low,single_dwelling_rebuild,75.0,zoning R2 | lot size band m | mixed zoning con...,The site has R2 zoning. The lot size band is m...
34140,4368209,622/5 HUTCHINSON WALK ZETLAND,R1,General Residential,mid_dev,0,1.900827e+03,m,0,None,...,None,1033.711910,0,800m_2km,0.000966,low,dual_occupancy,74.8,zoning R1 | lot size band m | no flood constra...,The site has R1 zoning. The lot size band is m...
38516,5372408,18 MISTFUL PARK ROAD GOULBURN,R2,Low Density Residential,low_dev,0,1.148708e+03,m,0,None,...,None,4230.393235,0,2km_5km,0.000236,low,granny_flat,71.5,zoning R2 | lot size band m | no flood constra...,The site has R2 zoning. The lot size band is m...
48605,7352126,7307/117 BATHURST STREET SYDNEY,SP5,Metropolitan Centre,high_dev,1,8.688177e+02,s,0,None,...,None,165.385472,1,within_800m,0.006010,low,land_bank_hold,73.0,zoning SP5 | lot size band s | mixed zoning co...,The site has SP5 zoning. The lot size band is ...
35605,4735263,164A YORK ROAD SOUTH PENRITH,R2,Low Density Residential,low_dev,0,1.623650e+03,m,0,None,...,None,3935.669808,0,2km_5km,0.000254,low,dual_occupancy,71.8,zoning R2 | lot size band m | no flood constra...,The site has R2 zoning. The lot size band is m...
29229,3395584,26/553 NEW CANTERBURY ROAD DULWICH HILL,SP2,Infrastructure,restricted,1,6.607532e+03,xl,0,None,...,None,1823.330053,0,800m_2km,0.000548,low,assembly_opportunity,50.6,zoning SP2 | lot size band xl | mixed zoning c...,The site has SP2 zoning. The lot size band is ...
4413,482527,4 MYALLIE AVENUE BAULKHAM HILLS,R2,Low Density Residential,low_dev,0,1.097722e+03,m,0,None,...,None,3974.959247,0,2km_5km,0.000252,low,granny_flat,71.5,zoning R2 | lot size band m | no flood constra...,The site has R2 zoning. The lot size band is m...
24674,2773457,2 GUERNSEY STREET SCONE,RE1,Public Recreation,restricted,1,8.274957e+03,xl,0,None,...,None,216.220834,1,within_800m,0.004604,low,low_rise_apartment,58.5,zoning RE1 | lot size band xl | mixed zoning c...,The site has RE1 zoning. The lot size band is ...
10783,1170814,11/1351 PITTWATER ROAD NARRABEEN,R3,Medium Density Residential,mid_dev,1,1.886666e+03,m,0,None,...,None,5904.552566,0,5km_10km,0.000169,low,land_bank_hold,65.0,zoning R3 | lot size band m | mixed zoning con...,The site has R3 zoning. The lot size band is m...
21421,2349167,7 KALGOORLIE STREET WILLOUGHBY,R2,Low Density Residential,low_dev,1,1.145395e+03,m,0,None,...,None,1428.921945,0,800m_2km,0.000699,low,granny_flat,79.0,zoning R2 | lot size band m | mixed zoning con...,The site has R2 zoning. The lot size band is m...
